# High Fidelity Data Ingestion for Oracle 23ai

**Copyright 2026, Denis Rothman**

This notebook serves as a production-ready template for Oracle DBAs and AI Engineers to implement **High Fidelity Data Ingestion** pipelines for Retrieval-Augmented Generation (RAG).

It moves beyond simple text storage by implementing a dual-stream ingestion process that handles both **Knowledge** (raw evidence) and **Context** (behavioral blueprints). The workflow demonstrates how to transform unstructured data into high-dimensional vectors and store them securely in Oracle Database 23ai.

### Notebook Highlights:

* **Use Case:** A **"Cyber-Incident Command Center"** scenario. We retrieve a complete security breach dataset—containing server logs, Slack transcripts, and phishing reports—to demonstrate how AI can correlate disparate data sources.
* **Infrastructure:** Establishes secure connectivity using Oracle Wallets and the `oracledb` thin client.
* **Data Pipeline:**
    * **Data Acquisition:** Secure retrieval of 7 distinct "Evidence" files from a remote repository.
    * **Transformation:** Robust text chunking (via `tiktoken`) and vector embedding (via OpenAI `text-embedding-3-small`).
    * **Loading:** Efficient batch insertion into Oracle Vector tables (`KNOWLEDGE_STORE` and `CONTEXT_LIBRARY`).
* **Validation:** Includes a test to perform semantic similarity searches (e.g., *"What happened to the admin database?"*) immediately after ingestion to verify index health.

## ⚠️ Instructions: Data Ingestion Workflow

**Crucial Step:** This notebook performs **Data Ingestion** (INSERT operations). To ensure data integrity and avoid errors, please follow this strict workflow:

1.  **Prerequisites:**
    * Ensure you have successfully ran the **`1_DBA_Oracle_Management.ipynb`** notebook with `create_tables = True`.
    * Verify that the `CONTEXT_LIBRARY` and `KNOWLEDGE_STORE` tables exist in your database.

2.  **Run Frequency:**
    * **Run this notebook ONLY ONCE.**
    * Running it multiple times without cleaning the database will cause **Primary Key Collisions** (`ORA-00001: unique constraint violated`) because the code attempts to insert duplicate IDs.

3.  **Troubleshooting (If the Insert Crashes):**
    * If you encounter an error during the upload (or if you need to restart the process), you must **clean the database first**.
    * **Action:** Go back to **`1_DBA_Oracle_Management.ipynb`**.
    * **Config:** Set `empty_tables = True`.
    * **Execute:** Run the notebook to truncate (wipe) the tables.
    * **Retry:** Return here and run this notebook again.

### ℹ️ Note: Shared Schema Management

Please be aware that the **Schema Manager** for this Chapter and notebook is also in**`Chapter03/1_DBA_Oracle_Management_V2.ipynb`**.

It manages the **entire** vector schema, encompassing both:
* **General RAG Tables** (from Chapter 2: `CONTEXT_LIBRARY`, `KNOWLEDGE_STORE`)
* **Hybrid RAG Tables** (from Chapter 3: `CANDIDATE_POOL`, `RECRUITMENT_RULES`)

**Action:** If you need to **delete** the tables to start over or clean up your environment, you must open **`Chapter03/1_DBA_Oracle_Management_V2.ipynb`**, set `empty_tables = True`, and run it. The Ingestion notebooks (starting with `2_...`) are for adding data only.



# 1.Installation and Setup

## 1.3.Installing OpenAI

In some cases, such as in this notebook for a Google Colab VM, OpenAI installation cells must be run first to avoid dependency conflicts.

In [1]:
#!pip install tqdm==4.67.1 --upgrade
!pip install openai==2.14.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.5 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.30.0
    Uninstalling openai-2.30.0:
      Successfully uninstalled openai-2.30.0


In [2]:
# Imports and API Key Setup
# We will use the OpenAI library to interact with the LLM and Google Colab's
# secret manager to securely access your API key.

import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    api_key = userdata.get("API_KEY")
    if not api_key:
        raise userdata.SecretNotFoundError("API_KEY not found.")

    # Set environment variable for downstream tools/libraries
    os.environ["OPENAI_API_KEY"] = api_key

    # Create client (will read from OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI API key loaded and environment variable set successfully.")

except userdata.SecretNotFoundError:
    print('Secret "API_KEY" not found.')
    print('Please add your OpenAI API key to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the API key: {e}")

# Configuration
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 # Dimension for text-embedding-3-small
GENERATION_MODEL = "gpt-5.2"

OpenAI API key loaded and environment variable set successfully.


In [3]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

GITHUB_TOKENAPI private token loaded successfully.


## DbA Parameters


## 1.1 Oracle environment Setup & Wallet Management

This step prepares the runtime environment by connecting to Google Drive and ensuring the Oracle Wallet is available. The Oracle Wallet contains the SSL/TLS certificates and configuration files (tnsnames.ora, sqlnet.ora) necessary for a secure connection to the Oracle Autonomous Database.

Google Drive Mount: Maps your personal Drive to /content/drive, allowing the notebook to read credentials and configuration files stored externally.

Wallet Unzipping: If unzip_wallet is set to True, the script searches for the wallet ZIP file in the specified path and extracts it. This ensures the connection drivers can locate the required security certificates.

Path Definition: Sets wallet_path to the directory where the unzipped configuration files reside, which will be passed to the Oracle driver in subsequent steps.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
wallet_path = "/content/drive/MyDrive/oracle_wallet"

Mounted at /content/drive


## 1.2 Install Oracle Database Driver
This command installs the oracledb Python driver, which is the renamed and modernized successor to the legacy cx_Oracle driver. It serves as the bridge between this Python notebook and the Oracle Database.

Version Pinning (==3.4.1): The version is explicitly fixed to 3.4.1 to ensure stability and reproducibility. In production DBA scripts, pinning versions prevents unexpected updates or API changes from breaking the automation workflow.

Thin Client Mode: By default, this driver operates in "Thin" mode, meaning it communicates directly with the database using pure Python code without requiring local Oracle Client libraries (Instant Client).

In [5]:
!pip install oracledb==3.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.0 MB/s eta 0:00:00


In [6]:
import oracledb
print(oracledb.__version__)

3.4.1


### Establish Secure Database Connection
This step handles the authentication and connection to the Oracle 23ai instance. It is designed to adhere to security best practices by strictly separating code from credentials.

External Credential Management: Instead of hardcoding sensitive passwords directly into the notebook (which is a security risk), the script reads a credentials.txt file stored securely on Google Drive.

Credential Parsing: The read_key_value_file helper function parses the external file to retrieve the username, password, Wallet password, and DSN (Data Source Name).

Connection Initialization: The oracledb.connect() method establishes the session using the extracted credentials and the local Wallet configuration path.

Connectivity Test: A simple "Heartbeat" query (SELECT ... FROM DUAL) is executed immediately to verify that the connection is active and stable before proceeding.

In [7]:
from google.colab import userdata
oracle_dsn = userdata.get('oracle_dsn')

In [8]:
import os

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", oracle_dsn)
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

('Connected!',)


## 1.4. Additional imports

In [9]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken                                                         # tokenization
from tenacity import retry, stop_after_attempt, wait_random_exponential # to retry functions
import re
import textwrap
from IPython.display import display, Markdown
import copy

# 2.Data Preparation: The Context Library (Procedural RAG)

### 2.1 Initialize Staging Directory
This step creates a local directory to act as a staging area for our raw data. In a real-world scenario, this might represent a secure S3 bucket or a network drive where logs and reports are aggregated during an incident. We will store our seven "evidence" files here before processing them into the Oracle database.

In [10]:
# Create a directory to store our source documents (Evidence)
import os

if not os.path.exists("incident_documents"):
    os.makedirs("incident_documents")
    print("Directory 'incident_documents' created.")
else:
    print("Directory 'incident_documents' already exists.")

Directory 'incident_documents' created.


## 2.2 Knowledge Store (The Evidence)

In this section, we retrieve the **evidence** files in a zip and unzip them. These represent the raw, unstructured data that an organization possesses but often struggles to correlate.

### 2.3 Data Acquisition: Secure Remote Ingestion

We begin by establishing a conditional connection to the remote repository. In a development cycle, we often rerun notebooks multiple times. Redownloading the same immutable artifacts consumes unnecessary bandwidth and time. Therefore, we introduce a boolean control flag `retrieve_evidence`. This allows the architect to toggle the ingestion process on or off directly from the notebook interface.

**Operational Scenarios**

This conditional logic addresses the two primary states of our cloud environment:

1.  **Case 1: The VM is Active (Rerun Cycle)**
    If the notebook has already been executed, the `incident_documents` directory persists in the local storage. By setting `retrieve_evidence = False`, the `curl` and `unzip` commands are skipped entirely. The pipeline proceeds using the existing files, preserving bandwidth.

2.  **Case 2: The Safety Path (Missing Data)**
    If the VM has been reset (Cold Start) and the flag is left on `False`, the local storage will be empty. Our pipeline includes a "Validation Gate" in the next section (`if os.path.exists`) to catch this gracefully, printing a warning rather than crashing.

In [11]:
retrieve_evidence = True #@param {type:"boolean"}

In [12]:
if retrieve_evidence==True:
  curl -L -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/enterprise_data/incident_evidence/evidence.zip" -o evidence.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   112    0   112    0     0    470      0 --:--:-- --:--:-- --:--:--   472


In [13]:
import time
if retrieve_evidence==True:
  time.sleep(10)
  !unzip -o evidence.zip -d incident_documents/

Archive:  evidence.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of evidence.zip or
        evidence.zip.zip, and cannot find evidence.zip.ZIP, period.


## 2.3.The Context Library(the instructions)

This section defines the Semantic Blueprints. While the Knowledge Store contains the facts (logs, reports), these blueprints contain the behavior.

We are creating three distinct "personas" for the AI:

Public Relations Manager: Generates reassuring, non-technical updates for customers.

Lead Engineer: Generates detailed, technical Root Cause Analysis (RCA) reports.

Legal Counsel: Generates formal risk assessments based on compliance policies.

In [14]:
# We define the Semantic Blueprints derived from our Cyber-Incident use case.
# The 'description' allows the Librarian agent to select the right format
# (e.g., "Draft a PR statement" vs "Analyze the root cause").

context_blueprints = [
    {
        "id": "blueprint_public_pr",
        "description": "A blueprint for drafting public-facing statements or press releases regarding service outages or security incidents. The goal is to reassure customers without admitting specific technical liabilities until confirmed.",
        "blueprint": json.dumps({
              "scene_goal": "Manage public perception and reassure customers.",
              "style_guide": "Use a professional, empathetic, but vague tone. Avoid technical jargon like 'SQL Injection' or 'Hacks'. Focus on 'Security Maintenance' or 'Unplanned Outage'.",
              "structure": ["Acknowledgment of Issue", "Current Status (Investigating)", "Estimated Resolution Time", "Apology"],
              "instruction": "Draft a public statement based on the provided facts. Do not reveal sensitive internal details (IPs, specific vulnerabilities)."
            })
    },
    {
        "id": "blueprint_technical_rca",
        "description": "A blueprint for creating a technical Root Cause Analysis (RCA) for engineering and security teams. Focuses on specific timestamps, error codes, vulnerabilities (CVEs), and remediation steps.",
        "blueprint": json.dumps({
              "scene_goal": "Provide a precise technical post-mortem of the incident.",
              "style_guide": "Clinical, precise, and fact-based. Use active voice. Explicitly reference log timestamps, error codes (ORA-XXXX), and IP addresses.",
              "structure": ["Incident Timeline", "Root Cause (The Vulnerability)", "Attack Vector", "Remediation (The Fix)"],
              "instruction": "Synthesize the provided logs and chat transcripts into a formal RCA document."
            })
    },
    {
        "id": "blueprint_legal_compliance",
        "description": "A blueprint for assessing legal liability and data breach notification obligations. It compares incident facts against internal privacy policies (GDPR/CCPA) to determine if a regulatory filing is required.",
        "blueprint": json.dumps({
              "scene_goal": "Determine legal risk and reporting requirements.",
              "style_guide": "Formal, risk-averse, and citation-heavy. Reference specific clauses from the provided Policy Documents.",
              "structure": ["Incident Summary", "Data Impact Assessment (Tier 1-4)", "Policy Violation Check", "Required Actions (Notification Yes/No)"],
              "instruction": "Analyze the incident against the 'Internal Data Privacy & Compliance Policy'. Determine if the compromised data requires external reporting."
            })
    }
]

print(f"\nPrepared {len(context_blueprints)} context blueprints.")


Prepared 3 context blueprints.


# 3.Updating the Data Loading and Processing Logic


This script iterates through the incident_documents directory we created in step 2.1. It reads every .txt file—logs, transcripts, and reports and loads them into a Python dictionary called knowledge_base.

This mimics a real-world ETL (Extract, Transform, Load) process where a DBA might script the ingestion of flat files from a server directory before processing them into the database.

In [15]:
# 3.Load Evidence into Memory
# -------------------------------------------------------------------------
# Load all documents from our new directory
knowledge_base = {}
doc_dir = "incident_documents"  # Updated to match our new use case

if os.path.exists(doc_dir):
    for filename in os.listdir(doc_dir):
        if filename.endswith(".txt"):
            with open(os.path.join(doc_dir, filename), 'r') as f:
                knowledge_base[filename] = f.read()

    print(f"📚 Loaded {len(knowledge_base)} documents into the knowledge base.")
else:
    print(f"❌ Directory '{doc_dir}' not found. Please run the document creation cells above.")

📚 Loaded 0 documents into the knowledge base.


# 4.Helper Functions for Chunking and Embedding

Before we can upload data to Oracle, we need two critical utilities:

chunk_text: A function to break large documents (like the "Patch Notes") into smaller, manageable pieces that fit within the AI's context window. We use tiktoken to ensure we split by tokens, not just characters.

get_embeddings_batch: A function that sends text to OpenAI's API and returns the vector representation (VECTOR(1536)). This is the bridge between human language and the Oracle Vector database.

In [16]:
# 4. Helper Functions for Chunking and Embedding
# -------------------------------------------------------------------------

# Initialize tokenizer for robust, token-aware chunking
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=400, overlap=50):
    """
    Chunks text based on token count with overlap.
    Best practice for RAG: Overlap ensures context isn't lost at the cut.
    """
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = tokenizer.decode(chunk_tokens)
        # Basic cleanup to remove excessive newlines
        chunk_text = chunk_text.replace("\n", " ").strip()
        if chunk_text:
            chunks.append(chunk_text)
    return chunks

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    """
    Generates embeddings for a batch of texts using OpenAI, with exponential backoff retries.
    """
    # OpenAI recommends replacing newlines with spaces for best embedding results
    texts = [t.replace("\n", " ") for t in texts]

    # Call the OpenAI API
    response = client.embeddings.create(input=texts, model=model)

    # Return just the list of embedding vectors
    return [item.embedding for item in response.data]

# 5.Processing and uploading the data to Oracle

In [ ]:
#@title 5.1.Processing and uploading Context Library to Oracle
import oracledb

print("\nProcessing and uploading Context Library to Oracle...")

# 1. Initialize the cursor
cursor = connection.cursor()

# 2. CRITICAL: Tell Oracle that the "embedding" list is actually a VECTOR type
# Without this, the driver thinks the list is a PL/SQL array (causing ORA-01484)
cursor.setinputsizes(embedding=oracledb.DB_TYPE_VECTOR)

vectors_context = []
for item in tqdm(context_blueprints):
    embedding = get_embeddings_batch([item['description']])[0]
    vectors_context.append({
        "id": item['id'],
        "values": embedding,
        "metadata": {
            "description": item['description'],
            "blueprint_json": item['blueprint']
        }
    })

if vectors_context:
    for item in vectors_context:
        cursor.execute("""
            INSERT INTO context_library (id, description, blueprint_json, embedding)
            VALUES (:id, :description, :blueprint_json, :embedding)
        """,
        {
            "id": item["id"],
            "description": item["metadata"]["description"],
            "blueprint_json": item["metadata"]["blueprint_json"],
            "embedding": item["values"]   # Now correctly interpreted as VECTOR
        })

    connection.commit()
    print(f"Successfully uploaded {len(vectors_context)} context vectors to Oracle.")

In [18]:
#@title 5.2.Processing and uploading Knowledge Base to Oracle
import oracledb

print("\nProcessing and uploading Knowledge Base to Oracle...")

# 1. Initialize the cursor explicitly to ensure it exists
cursor = connection.cursor()

# 2. Tell Oracle that the "embedding" bind variable is a VECTOR type
cursor.setinputsizes(embedding=oracledb.DB_TYPE_VECTOR)

batch_size = 100
total_vectors_uploaded = 0

for doc_name, doc_content in knowledge_base.items():
    print(f"  - Processing document: {doc_name}")

    knowledge_chunks = chunk_text(doc_content)

    for i in tqdm(range(0, len(knowledge_chunks), batch_size), desc=f"  Uploading {doc_name}"):
        batch_texts = knowledge_chunks[i:i+batch_size]
        batch_embeddings = get_embeddings_batch(batch_texts)

        for j, embedding in enumerate(batch_embeddings):
            chunk_id = f"{doc_name}_chunk_{total_vectors_uploaded + j}"

            cursor.execute("""
                INSERT INTO knowledge_store (id, source, text, embedding)
                VALUES (:id, :source, :text, :embedding)
            """,
            {
                "id": chunk_id,
                "source": doc_name,
                "text": batch_texts[j],
                "embedding": embedding
            })

        connection.commit()

    total_vectors_uploaded += len(knowledge_chunks)

print(f"\nSuccessfully uploaded {total_vectors_uploaded} knowledge vectors to Oracle.")


Processing and uploading Knowledge Base to Oracle...

Successfully uploaded 0 knowledge vectors to Oracle.


# 6.Final Verification


This step performs a live test of the vector index. We use a specific question—"What happened to the admin database?"—to verify that the database can semantically link a natural language query to the raw evidence files (like the Server Logs and Slack Transcripts) we just uploaded.

Embedding: The query is converted into a vector using the same model as our data.

VECTOR_DISTANCE: Oracle calculates the distance between our query vector and every stored document vector.

Ranking: The results are ordered by the <-> operator (Euclidean distance), bringing the most relevant "evidence" to the top.

In [19]:
import oracledb

# Prepare a test query
test_query = "What happened to the admin database?"
query_embedding = get_embeddings_batch([test_query])[0]

cursor.setinputsizes(query_vec=oracledb.DB_TYPE_VECTOR)

cursor.execute("""
    SELECT id, source, text,
           VECTOR_DISTANCE(embedding, :query_vec) AS distance
    FROM knowledge_store
    ORDER BY embedding <-> :query_vec
    FETCH FIRST 5 ROWS ONLY
""",
{
    "query_vec": query_embedding
})

results = cursor.fetchall()

print("\nTop 5 vector search results:\n")
for r in results:
    text_clob = r[2]              # this is a CLOB
    text_str = text_clob.read()   # convert to Python string

    print(f"ID: {r[0]}")
    print(f"Source: {r[1]}")
    print(f"Text snippet: {text_str[:120]}...")
    print(f"Distance: {r[3]}")
    print("-" * 60)


Top 5 vector search results:



# 7.System Status Summary

In [20]:
print("\n=== Oracle Vector Table Summary ===\n")

# --- 1. Table existence ---
cursor.execute("""
SELECT table_name
FROM user_tables
WHERE table_name IN ('CONTEXT_LIBRARY', 'KNOWLEDGE_STORE')
""")
print("Tables present:", [t[0] for t in cursor.fetchall()])


# --- 2. Schema for CONTEXT_LIBRARY ---
print("\n--- CONTEXT_LIBRARY Schema ---")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'CONTEXT_LIBRARY'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)


# --- 3. Schema for KNOWLEDGE_STORE ---
print("\n--- KNOWLEDGE_STORE Schema ---")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'KNOWLEDGE_STORE'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)


# --- 4. Row counts ---
cursor.execute("SELECT COUNT(*) FROM context_library")
context_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM knowledge_store")
knowledge_count = cursor.fetchone()[0]

print(f"\nRows in CONTEXT_LIBRARY: {context_count}")
print(f"Rows in KNOWLEDGE_STORE: {knowledge_count}")


# --- 5. Sample rows (non‑vector fields only) ---
print("\n--- Sample from CONTEXT_LIBRARY ---")
cursor.execute("""
SELECT id, SUBSTR(description, 1, 120)
FROM context_library
FETCH FIRST 3 ROWS ONLY
""")
for row in cursor.fetchall():
    print(row)


print("\n--- Sample from KNOWLEDGE_STORE ---")
cursor.execute("""
SELECT id, source, SUBSTR(text, 1, 120)
FROM knowledge_store
FETCH FIRST 3 ROWS ONLY
""")
for row in cursor.fetchall():
    print(row)

print("\n=== Summary Complete ===")



=== Oracle Vector Table Summary ===

Tables present: ['CONTEXT_LIBRARY', 'KNOWLEDGE_STORE']

--- CONTEXT_LIBRARY Schema ---
('ID', 'VARCHAR2', 200)
('DESCRIPTION', 'CLOB', 4000)
('BLUEPRINT_JSON', 'CLOB', 4000)
('EMBEDDING', 'VECTOR', 8200)

--- KNOWLEDGE_STORE Schema ---
('ID', 'VARCHAR2', 200)
('SOURCE', 'VARCHAR2', 200)
('TEXT', 'CLOB', 4000)
('EMBEDDING', 'VECTOR', 8200)

Rows in CONTEXT_LIBRARY: 3
Rows in KNOWLEDGE_STORE: 0

--- Sample from CONTEXT_LIBRARY ---
('blueprint_public_pr', <oracledb.lob.LOB object at 0x7c46adf9e8a0>)
('blueprint_technical_rca', <oracledb.lob.LOB object at 0x7c46ac169370>)
('blueprint_legal_compliance', <oracledb.lob.LOB object at 0x7c46ac169010>)

--- Sample from KNOWLEDGE_STORE ---

=== Summary Complete ===
